# Fase 03 : Entorno de simulación
**Institución:** Universidad Católica Boliviana "San Pablo"  
**Programa:** Maestría en Ciencia de Datos  
**Autor:** Carlos Andrés Pérez Guzmán  
**Fecha:** Septiembre 2026  

---
**Entorno y Objetivos del Notebook**
> **Nota:** Preparación del entorno de simulación

* **Fase del Proyecto:** Tercera fase fase
* **Core Tech Stack:** `Python` (Programación general para ciencia de datos)
---

<h3 style="color:#05856d">1. Librerías</h3>
<hr>

In [33]:
import polars as pl
from pyprojroot import here
import s_01_functions as fn_01

<h3 style="color:#05856d">2. Carga de información</h3>
<hr>

In [34]:
df_02 = pl.read_parquet(source = here()/'01_cleaned_data'/'df_02.parquet')
df_02.glimpse()

Rows: 10580
Columns: 11
$ bottle_volume_ml   <cat> 1750, 1750, 1750, 1750, 1750, 1750, 1750, 1750, 1750, 1750
$ category_name      <cat> CANADIAN WHISKIES, CANADIAN WHISKIES, CANADIAN WHISKIES, CANADIAN WHISKIES, CANADIAN WHISKIES, CANADIAN WHISKIES, CANADIAN WHISKIES, CANADIAN WHISKIES, CANADIAN WHISKIES, CANADIAN WHISKIES
$ im_desc            <cat> BLACK VELVET, BLACK VELVET, BLACK VELVET, BLACK VELVET, BLACK VELVET, BLACK VELVET, BLACK VELVET, BLACK VELVET, BLACK VELVET, BLACK VELVET
$ item_no            <i64> 11788, 11788, 11788, 11788, 11788, 11788, 11788, 11788, 11788, 11788
$ ordered_on        <date> 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18, 2026-02-18
$ pack               <i64> 6, 6, 6, 6, 6, 6, 6, 6, 6, 6
$ period_ordered_on  <cat> 2026-02, 2026-02, 2026-02, 2026-02, 2026-02, 2026-02, 2026-02, 2026-02, 2026-02, 2026-02
$ product_code       <str> 'PRD_000201', 'PRD_000201', 'PRD_000201', 'PRD_000201', 'PRD_000201

<h3 style="color:#05856d">3. Preparación de datos</h3>
<hr>

Seríe de tiempo de la demanda

In [35]:
df_demanda = (
    df_02
    .group_by('ordered_on')
    .agg(pl.sum('sales_bottles').alias('demanda_diaria'))
    .sort('ordered_on')
    .with_columns( pl.col('ordered_on').dt.weekday().alias('dia_semana'))
    )

print(f"Serie de tiempo diaria creada: {df_demanda.height} días de operación.")

df_demanda.head()

Serie de tiempo diaria creada: 115 días de operación.


ordered_on,demanda_diaria,dia_semana
date,i64,i8
2026-01-02,1026,5
2026-01-04,114,7
2026-01-05,833,1
2026-01-06,1137,2
2026-01-07,644,3


Extraer parámetros específicos para el modelo base

In [37]:
# Cantidad de botellas por pack
pack_u = df_02['pack'][0]

# Precio unitario promedio
precio_u = (df_02['sales_dollars'] / df_02['sales_bottles']).mean()

# Crear un DataFrame tempoaral de parámetros
df_temp = pl.DataFrame({'precio_medio': [precio_u], 'pack': [pack_u]})
df_temp

precio_medio,pack
f64,i64
17.237854,6


Ejecutar el modelo base

In [42]:
eoq_cajas = fn_01.eoq_baseline(
    df_demanda_diaria = df_demanda, 
    df_param = df_temp, 
    dias_year = 365, 
    costo_pedido_S = 50.0, 
    tasa_mantenimiento_i = 0.20 )

--------------------------------------------------------------
RESULTADOS DEL MODELO BASE (EOQ)
--------------------------------------------------------------
Demanda Anual Estimada (D)       : 508,086 botellas
Costo de Mantenimiento Anual (H) : $ 3.45 por botella
Costo Fijo por Pedido (S)        : $ 50.00
EOQ Óptimo Teórico               : 3,839 botellas
EOQ Ajustado a "packs"           : 640 packs (de 6 botellas) 
--------------------------------------------------------------
